# Phase 3 — Factor Study: Value Factor

**Purpose:** Demonstrate `factor_study()` on `ValueFactorStrategy` over a synthetic market.
Shows IC summary, quantile returns, and block bootstrap confidence interval.

No live data required — all cells use `build_synthetic_market()`.

In [1]:
# Cell 1: Imports and environment setup
import pathlib
import sys
from datetime import date

import ah_research
from ah_research.analysis import FactorReport, factor_study
from ah_research.strategies import ValueFactorStrategy

# Make tests/fixtures importable
sys.path.insert(0, str(pathlib.Path().resolve() / "tests"))
from fixtures.phase2.synthetic_market import build_synthetic_market

code_version = getattr(ah_research, "__version__", "dev")
print(f"Python : {sys.version.split()[0]}")
print(f"ah-research version: {code_version}")

Python : 3.12.10
ah-research version: 0.0.1


In [2]:
# Cell 2: Build synthetic market (2 years, 5 symbols)
STUDY_START = date(2022, 1, 1)
STUDY_END = date(2023, 12, 31)
SYMBOLS = ["600000.SH", "000001.SZ", "600519.SH", "600036.SH", "601318.SH"]

repo = build_synthetic_market(
    start=STUDY_START,
    end=STUDY_END,
    symbols=SYMBOLS,
)

print(f"Synthetic market: {len(SYMBOLS)} symbols, {STUDY_START} to {STUDY_END}")

Synthetic market: 5 symbols, 2022-01-01 to 2023-12-31


In [3]:
# Cell 3: Run factor study on ValueFactorStrategy
strategy = ValueFactorStrategy()

report: FactorReport = factor_study(
    strategy,
    repo,
    start=STUDY_START,
    end=STUDY_END,
    n_quantiles=5,
    ic_horizons=[5, 20],
    sector_neutral=True,
    bootstrap_n_resamples=200,
    rebalance="M",
    random_seed=42,
)

print(f"Rebalance dates used : {report.n_rebalance_dates}")
print(f"Sector neutralized   : {report.sector_neutralized}")
print(f"Avg names per period : {report.universe_summary.get('avg_n_names', 'n/a')}")

Rebalance dates used : 24
Sector neutralized   : True
Avg names per period : 5


/Users/brian_huang/repos/ah-research/src/ah_research/analysis/factor_study.py:51: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic, _ = spearmanr(paired.iloc[:, 0].values, paired.iloc[:, 1].values)


In [4]:
# Cell 4: IC summary table
print("=== IC Summary (Spearman, Newey-West t-stat) ===")
print(report.ic_summary.to_string())

=== IC Summary (Spearman, Newey-West t-stat) ===
         mean_ic  nw_t_stat  nw_p_value  ir
horizon                                    
5            NaN        NaN         NaN NaN
20           NaN        NaN         NaN NaN


In [5]:
# Cell 5: Quantile returns summary
print("=== Quantile Returns Summary ===")
if not report.quantile_summary.empty:
    print(report.quantile_summary.to_string())
else:
    print("(no quantile data — too few symbols for 5 buckets)")

print()
print("=== Block Bootstrap: Q5 - Q1 Long/Short ===")
bs = report.bootstrap_q5_minus_q1
print(f"  Mean    : {bs.get('mean', float('nan')):.4f}")
print(f"  95% CI  : [{bs.get('ci_low', float('nan')):.4f}, {bs.get('ci_high', float('nan')):.4f}]")
print(f"  p-value : {bs.get('p_value', float('nan')):.4f}")

=== Quantile Returns Summary ===
                cagr    sharpe  max_drawdown
quantile                                    
Q1         -0.009448  0.280620     -0.234905
Q2         -0.052603 -0.334339     -0.208035
Q3         -0.049726 -1.721795     -0.305523
Q4          0.031974  0.275502     -0.216746
Q5         -0.062844 -1.402410     -0.139919
long_short -0.134210 -1.126121     -0.420868

=== Block Bootstrap: Q5 - Q1 Long/Short ===
  Mean    : -0.0079
  95% CI  : [nan, nan]
  p-value : nan


In [6]:
# Cell 6: IC decay across horizons
print("=== IC Decay (mean IC by horizon) ===")
print(report.ic_decay.to_string())
print()
print("=== Quantile Returns (first 5 dates) ===")
print(report.quantile_returns.head().to_string())

=== IC Decay (mean IC by horizon) ===
horizon
5    NaN
20   NaN

=== Quantile Returns (first 5 dates) ===
                  Q1        Q2        Q3        Q4        Q5  long_short
date                                                                    
2022-01-31 -0.039295 -0.017885 -0.118937 -0.108068 -0.029277    0.010018
2022-02-28  0.025941 -0.028310 -0.028199  0.109405 -0.056601   -0.082542
2022-03-31 -0.149920  0.044483 -0.121384 -0.042139  0.045900    0.195820
2022-05-31 -0.099973 -0.131171  0.024919 -0.034289 -0.012045    0.087929
2022-06-30  0.079035  0.096164 -0.202664  0.020711  0.020159   -0.058877


In [7]:
# Cell 7: Reproducibility block
import subprocess

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    git_sha = "unknown"

print("=== Reproducibility ===")
print(f"Code version : {code_version}")
print(f"Git SHA      : {git_sha}")
print(f"Study window : {STUDY_START} to {STUDY_END}")
print(f"Symbols      : {SYMBOLS}")
print("Random seed  : 42")
print("Data source  : synthetic (build_synthetic_market)")

=== Reproducibility ===
Code version : 0.0.1
Git SHA      : d786f88
Study window : 2022-01-01 to 2023-12-31
Symbols      : ['600000.SH', '000001.SZ', '600519.SH', '600036.SH', '601318.SH']
Random seed  : 42
Data source  : synthetic (build_synthetic_market)


---
> **DISCLAIMER:** Results are based on synthetic / randomly generated data. This is a historical backtest demonstration only and does not constitute investment advice. Past performance does not guarantee future results.